# MLB Offensive Production Analysis

This notebook studies same-season associations between hitter traits and offensive production for model-eligible player-seasons in the cached 2024–2026 dataset. It is not a causal analysis or a next-season forecast.

Three parts:

1. **Rank** swing and stance traits by correlation with production.
2. **Check overlap** with core stats (barrels, EV, hard-hit, K%, BB%) and residualize.
3. **Reconstruct** same-season wRC+ from core stats, then from core plus traits.

- **Outcome:** FanGraphs **wRC+**, with Statcast **xwOBA** as a second check (results vs expected contact).
- **Traits:** bat tracking and stance/setup (bat speed, attack angle, distance off the plate, and related Statcast fields).
- **Excluded from the ranking:** barrels, exit velocity, OPS, and wOBA. Those are production itself or batted-ball results sitting downstream of the swing; putting them on the list would crowd out the traits we actually want to rank.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from mlb_offense.analysis import (
    AnalysisConfig,
    bootstrap_model_difference,
    clustered_ols_coefficients,
    coverage_by_season,
    evaluate_models,
    feature_sets,
    format_trait_ranking,
    grouped_permutation_importance,
    load_analysis_data,
    missingness_by_season,
    plot_clustered_coefficients,
    plot_missingness,
    plot_outcome_distributions,
    plot_permutation_importance,
    plot_predictions,
    plot_raw_vs_residualized,
    plot_reconstruction_mae,
    plot_residual_diagnostics,
    plot_trait_block_increment,
    plot_trait_core_heatmap,
    plot_trait_ranking,
    rank_hitter_traits,
    residualized_trait_associations,
    trait_core_correlations,
)

DATA_PATH = Path('data/processed/mlb_offense_2024_2026.parquet')
OUTPUT_DIR = Path('data/analysis')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

config = AnalysisConfig(
    min_pa=100,
    min_competitive_swings=50,
    outer_splits=5,
    inner_splits=3,
    bootstrap_iterations=500,
)
data = load_analysis_data(DATA_PATH, config)
blocks = feature_sets(data)
print(f'{len(data):,} eligible player-seasons across {data.player_id.nunique():,} hitters')

## Sample coverage and descriptive outcomes

The local thresholds are intentionally applied after data collection. This keeps the raw source responses intact and makes sensitivity checks straightforward.

In [ ]:
display(coverage_by_season(data).style.format(precision=3))
display(
    data.groupby('season')[['pa', 'competitive_swings', 'wrc_plus', 'ops_plus', 'woba', 'xwoba']]
    .describe()
)

## Part 1: Which swing and stance traits track production?

The ranking below is the simple question: among **how a hitter swings and sets up**, which fields move with offensive production?

Each row is an eligible player-season (`PA >= 100`, `competitive swings >= 50`). Pearson *r* is linear association; Spearman *ρ* is the rank-based check. **wRC+** is observed run production. **xwOBA** is expected production from contact quality, so a trait that ranks high on both is not just riding BABIP noise.

### Why barrels, exit velocity, OPS, and wOBA are not on this list

Those four would dominate a correlation table, and they would do it for the wrong reason.

- **OPS and wOBA are production.** OPS is a counting-rate mix of OBP and SLG. wOBA is a linear-weights production metric. wRC+ is a scaled run-production index built from the same family of ingredients. Asking how much OPS or wOBA correlates with wRC+ is nearly tautological: you would be ranking the outcome against itself. They belong as *targets*, not as traits. xwOBA is used here only as a second outcome, never as a trait.
- **Barrels and exit velocity are batted-ball results, not swing traits.** A barrel is already a ball hit in a high-value exit-velo / launch-angle window. Average EV is how hard the ball came off the bat. Both sit *downstream* of the swing. If they were included, they would crowd out bat speed, attack angle, and stance because they are closer to the production mechanism. They return in Part 2 as the overlap check, and in Part 3 as the `core` block.

Hard-hit rate and average launch angle are kept off the ranking for the same downstream-contact reason. Plate-discipline rates (walks, strikeouts, chase) are also not swing/stance traits; they are approach outcomes and belong with the core stats.

What remains is the list that matches the original goal: bat speed, swing length, squared-up and blast rates, whiff, attack angle and direction, tilt, distance off the plate, depth in the box, and intercept.

In [ ]:
ranking = rank_hitter_traits(data)
ranking.to_csv(OUTPUT_DIR / 'trait_outcome_rankings.csv', index=False)

ranking_table = format_trait_ranking(ranking)
ranking_table.to_csv(OUTPUT_DIR / 'trait_ranking_wide.csv', index=False)
display(
    ranking_table.style.format({
        column: '{:.3f}' if 'n_' not in column else '{:.0f}'
        for column in ranking_table.columns
        if column not in {'trait', 'trait_label'}
    })
)

ranking_figure = plot_trait_ranking(ranking)
ranking_figure.savefig(OUTPUT_DIR / 'trait_ranking.png', dpi=160, bbox_inches='tight')
plt.show()

The ranking is descriptive association among eligible player-seasons, not a claim that any trait causes production.

Blast rate (Statcast’s combination of a fast swing and a squared-up contact) leads both lists. Fast-swing rate and average bat speed follow. Stance and intercept fields sit near zero. Correlations with xwOBA are stronger than with wRC+ for the top traits, which is what you would expect if those traits track contact quality more tightly than realized results.

Squared-up and blast rates are still *swing* descriptors, not batted-ball outcomes like barrels or EV. They stay in Part 1 for that reason. Part 2 checks how much of their ranking is already sitting in barrels, exit velocity, and strikeouts.

## Part 2: How much of that ranking is just barrels and strikeouts?

A high correlation in Part 1 does not mean the trait is adding information beyond ordinary contact quality. Fast swings that produce barrels show up as **bat speed** and as **barrel rate**. The same hitter also tends to walk and strike out at different rates.

This section keeps barrels, EV, and wOBA *out of the trait list*, but brings barrels, hard-hit rate, EV, K%, and BB% back as **controls** — the downstream stats the swing is supposed to produce.

1. A heatmap of Pearson *r* between swing/stance traits and those core stats.
2. A dumbbell chart: each trait’s raw *r* with wRC+, then the *r* that remains after both the trait and wRC+ are residualized on that core block.

If a trait’s bar collapses toward zero, its Part 1 ranking was mostly contact quality in disguise. If it holds, there is leftover association with production after barrels, EV, hard-hit, K%, and BB% are accounted for.

In [ ]:
overlap = trait_core_correlations(data)
overlap.to_csv(OUTPUT_DIR / 'trait_core_correlations.csv')
display(overlap.round(3))

overlap_figure = plot_trait_core_heatmap(overlap)
overlap_figure.savefig(OUTPUT_DIR / 'trait_core_heatmap.png', dpi=160, bbox_inches='tight')
plt.show()

residualized = residualized_trait_associations(data)
residualized.to_csv(OUTPUT_DIR / 'trait_residualized_associations.csv', index=False)
display(
    residualized[['trait_label', 'raw_r', 'residualized_r', 'n']].style.format({
        'raw_r': '{:.3f}',
        'residualized_r': '{:.3f}',
        'n': '{:.0f}',
    })
)

residualized_figure = plot_raw_vs_residualized(residualized)
residualized_figure.savefig(OUTPUT_DIR / 'trait_raw_vs_residualized.png', dpi=160, bbox_inches='tight')
plt.show()

Read the heatmap as collinearity, not as a second ranking. Blast rate and bat speed line up with barrels, hard-hit rate, and EV (blast vs hard-hit is about 0.9). Whiff rate lines up with strikeouts (~0.87). Walk rate barely moves with any swing trait. Stance and intercept stay quiet.

The dumbbell is the stricter check. Blast rate’s raw *r* with wRC+ is about 0.50; after residualizing on barrels, EV, hard-hit, K%, and BB% it is roughly zero. Bat speed and fast-swing rate collapse the same way. That is the complication: Part 1’s leaders were mostly contact quality in disguise. Leftover correlation is what Part 3 asks the models to find.

In [ ]:
figures = [
    plot_outcome_distributions(data),
    plot_missingness(data),
]
for number, figure in enumerate(figures, start=1):
    figure.savefig(OUTPUT_DIR / f'eda_{number}.png', dpi=160, bbox_inches='tight')
    plt.show()

display(missingness_by_season(data).sort_values(['season', 'missing_rate'], ascending=[True, False]).head(20))

## Part 3: Reconstructing same-season wRC+

Part 2 showed that most of the trait–production correlation collapses after barrels, EV, hard-hit, K%, and BB%. This section asks the same question as a prediction problem: once those core stats (plus age, handedness, and season) are already in the model, do swing and stance fields help reconstruct this season’s wRC+?

`core` is the conventional block. `core_plus_traits` adds the Part 1 swing/stance fields. OPS, wOBA, and xwOBA stay out of the predictors so the model cannot reconstruct wRC+ from a near-equivalent production formula.

The headline comparison is **elastic net, core vs core plus traits**, with a player-clustered bootstrap on the MAE difference. OLS, ridge, and a constrained random forest are shown only to check that the increment is not an artifact of one algorithm. Outer folds are grouped by hitter so a player’s seasons are never split across train and test. This is same-season association, not a next-year forecast.

In [ ]:
performance, predictions, selected_parameters = evaluate_models(data, config)
performance.to_csv(OUTPUT_DIR / 'grouped_cv_performance.csv', index=False)
predictions.to_parquet(OUTPUT_DIR / 'out_of_fold_predictions.parquet', index=False)
selected_parameters.to_json(OUTPUT_DIR / 'selected_parameters.json', orient='records', indent=2)

display(
    performance.style.format({
        'mae': '{:.2f}', 'rmse': '{:.2f}', 'r2': '{:.3f}',
        'calibration_intercept': '{:.2f}', 'calibration_slope': '{:.3f}',
        'pa_weighted_mae': '{:.2f}', 'pa_weighted_rmse': '{:.2f}',
        'pa_weighted_r2': '{:.3f}',
    })
)

reconstruction_figure = plot_reconstruction_mae(performance)
reconstruction_figure.savefig(OUTPUT_DIR / 'reconstruction_mae.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
comparison = bootstrap_model_difference(
    predictions,
    candidate_model='elastic_net:core_plus_traits',
    reference_model='elastic_net:core',
    metric='mae',
    iterations=config.bootstrap_iterations,
    random_state=config.random_state,
)
comparison.to_csv(OUTPUT_DIR / 'elastic_net_trait_increment_bootstrap.csv', index=False)
display(comparison.style.format({'difference': '{:.3f}', 'ci_lower': '{:.3f}', 'ci_upper': '{:.3f}'}))

increment_figure = plot_trait_block_increment(comparison)
increment_figure.savefig(OUTPUT_DIR / 'trait_mae_increment.png', dpi=160, bbox_inches='tight')
plt.show()

Core stats already do the work: guessing the mean is off by about 22 wRC+ points; elastic net on `core` is off by 14.5 (R² 0.55). Adding swing and stance traits trims MAE by **0.28 points** (bootstrap interval about −0.48 to −0.08). That increment is real and tiny, and it matches Part 2: once barrels, EV, and strikeouts are in the model, bat speed is mostly redundant.

The linear models all tell the same story. The random forest is worse, so the result is not “we needed a fancier algorithm.” Negative MAE difference means traits help; the interval does not include zero, but 0.28 wRC+ is not a scouting-relevant edge on its own.

In [ ]:
best_model = performance.iloc[0]['model']
prediction_figure = plot_predictions(predictions, best_model)
prediction_figure.savefig(OUTPUT_DIR / 'best_model_calibration.png', dpi=160, bbox_inches='tight')
plt.show()

residual_figure = plot_residual_diagnostics(predictions, best_model, data)
residual_figure.savefig(OUTPUT_DIR / 'best_model_residuals.png', dpi=160, bbox_inches='tight')
plt.show()

### Joint coefficients and permutation importance

These are descriptive, not a second ranking. Permutation importance asks what the forest uses to reconstruct wRC+. Standardized OLS coefficients ask what remains after every other predictor is in the model at once. Correlated traits can steal credit from each other; a negative bat-speed coefficient is collinearity with barrels and EV, not evidence that swinging faster hurts.

In [ ]:
coefficients = clustered_ols_coefficients(data, blocks['core_plus_traits'], config)
coefficients.to_csv(OUTPUT_DIR / 'clustered_ols_coefficients.csv', index=False)
display(coefficients.head(20).style.format({
    'coefficient': '{:.3f}', 'std_error': '{:.3f}', 'ci_lower': '{:.3f}',
    'ci_upper': '{:.3f}', 'p_value': '{:.4f}',
}))

coefficient_figure = plot_clustered_coefficients(coefficients)
coefficient_figure.savefig(OUTPUT_DIR / 'clustered_ols_coefficients.png', dpi=160, bbox_inches='tight')
plt.show()

importance = grouped_permutation_importance(data, blocks['core_plus_traits'], config)
importance.to_csv(OUTPUT_DIR / 'permutation_importance.csv', index=False)
display(importance.head(20).style.format({'importance_mean': '{:.3f}', 'importance_std': '{:.3f}'}))

importance_figure = plot_permutation_importance(importance)
importance_figure.savefig(OUTPUT_DIR / 'permutation_importance.png', dpi=160, bbox_inches='tight')
plt.show()

## Interpretation checklist

- Part 1 ranks swing and stance traits against production. It is association, not causation.
- Part 2 is the overlap check: if a trait’s correlation with wRC+ collapses after residualizing on barrels, EV, hard-hit, K%, and BB%, it was mostly contact quality.
- Part 3: core stats take MAE from ~22 to ~14.5. Traits add about 0.28 wRC+ of MAE. The bootstrap interval excludes zero, but the edge is small.
- A negative bat-speed OLS coefficient after barrels and EV are in the model is collinearity, not “bat speed hurts.”
- Check residual plots and calibration before trusting aggregate error statistics.
- Re-run with alternative PA and competitive-swing thresholds before making conclusions about the MLB hitter population.